# E1 · The early stopping that never stops

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Understand why a colleague, who turned on `early_stopping_rounds=30` and "monitors both curves" by passing training and validation to `eval_set`, always gets `best_iteration = 499` (the model uses all the rounds) and concludes that "early stopping is not needed, the model always improves". Find **which set XGBoost uses
to stop**, fix it and measure the difference on the test set.

## Dataset

`make_classification(5000, 12 features, 5 informative, flip_y=0.1)`; split 50% training, 25% validation, 25% test. Model: `XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=5, eval_metric="logloss", early_stopping_rounds=30)`.

## Educational purpose

Know a rule that is easy to miss: with several sets in `eval_set`, early stopping looks at **the last one**. Passing `[(X_val, y_val), (X_tr, y_tr)]` means stopping on the training curve, which always drops: the model never stops and overfits, with no error or warning.

## Task

1. Run the colleague's cell and observe `best_col` (the number of rounds used) and `bug_test_ll` (log-loss on the test set).
2. Diagnosis: save `which_set` (`"training"` or `"validation"`: the set on which XGBoost evaluates the stopping criterion) and `why_it_never_stops` (a short string). Check it with `evals_result()`: the `validation_0` curve is the validation one (first element), `validation_1` the training one (second); save `val_min_round` (round, from 1, at which the validation curve is at its minimum **in the colleague's run**).
3. Fix: put the validation set **last** in `eval_set` (`[(X_tr, y_tr), (X_val, y_val)]`, or only the validation set): save `ok_round` (rounds used) and `ok_test_ll` (log-loss on the test set).
4. Check: in the corrected run, the `argmin` of the validation curve (the last key of `evals_result()`) matches `best_iteration`; save `matches` (True/False). Save `gain` (`bug_test_ll − ok_test_ll`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

X, y = make_classification(5000, n_features=12, n_informative=5, class_sep=0.7, flip_y=0.1, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.5, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)


def new_model():
    return xgb.XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=5, eval_metric="logloss", early_stopping_rounds=30, n_jobs=-1, random_state=0)

## Your work

### The colleague's code

In [ ]:
m_bug = new_model()
m_bug.fit(X_tr, y_tr, eval_set=[(X_val, y_val), (X_tr, y_tr)], verbose=False)      # 'I monitor validation and training'
best_col = m_bug.best_iteration + 1
bug_test_ll = log_loss(y_te, m_bug.predict_proba(X_te)[:, 1])
print(f"rounds used: {best_col} out of 500 | test log-loss {bug_test_ll:.4f} -> 'early stopping is not needed, the model always improves'")

### Your diagnosis and the fix

In [ ]:
which_set = None
why_it_never_stops = None
val_min_round = None

In [ ]:
ok_round = None
ok_test_ll = None
matches = None
gain = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`res = m_bug.evals_result()`; the key `"validation_0"` is the first set of `eval_set`. In the fix the last key is the validation set: `list(m.evals_result())[-1]`.'):
    check(best_col >= 490, "the colleague sees that the model uses almost all the 500 rounds", "don't modify the colleague's code")
    check(which_set == "training" and isinstance(why_it_never_stops, str) and len(why_it_never_stops) > 10, "XGBoost uses the last set of eval_set, that is the training set, and you explain why", "XGBoost stops on eval_set[-1]")
    check(val_min_round is not None and 20 <= val_min_round <= 80, "the validation, in the colleague's run, had its minimum between 20 and 80 rounds: from there on the model was getting worse", "argmin of evals_result()['validation_0']['logloss'] + 1")
    check(ok_round is not None and ok_round < 100 and ok_test_ll < 0.42, "with the validation set last the model stops under 100 rounds and the test log-loss drops below 0.42", "eval_set=[(X_tr, y_tr), (X_val, y_val)]")
    check(matches is True and gain is not None and gain > 0.05, "the minimum of the validation curve matches best_iteration; the gain on the test log-loss exceeds 0.05", "bug_test_ll - ok_test_ll")
done()

---
The commented solution is in `solutions/E_find_the_bug/E1_the_early_stopping_that_never_stops.ipynb`. Open it after you pass the check, or after you have really tried.